# Contagion Simulations

For every bank in every quarter (2016 Q1 – 2023 Q4), apply a random equity shock
via `simulate_failure` and store the cascade results as parquet files.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from pathlib import Path

from src.data import load_data
from src.models import simulate_failure

PROJECT_ROOT = Path().resolve().parent
OUT_SIMULATION = PROJECT_ROOT / "src" / "data" / "simulation_features"
OUT_SIMULATION.mkdir(parents=True, exist_ok=True)

OUT_SIMULATION

In [ ]:
# Load all 32 quarters
quarters = {}

for year in range(2016, 2024):
    for q in range(1, 5):
        edges, nodes = load_data(year, q)
        quarters[(year, q)] = (edges, nodes)

In [ ]:
# --- configuration ---
N_SIMULATIONS = 500000   # simulations per quarter (change as needed)

simulation_summary_rows = []

for (year, q), (edges, nodes) in quarters.items():

    # Seeded RNG per quarter for reproducibility
    rng = np.random.default_rng(seed=year * 10 + q)
    bank_ids = nodes["index"].tolist()

    # Draw N random banks (with replacement) and N random shock fractions
    sampled_banks = rng.choice(bank_ids, size=N_SIMULATIONS, replace=True)
    shock_fracs   = rng.uniform(0.05, 0.60, size=N_SIMULATIONS)

    sim_rows = []
    for bank_id, shock_frac in zip(sampled_banks, shock_fracs):
        result = simulate_failure(
            int(bank_id), edges, nodes,
            mechanism="Exposure",
            initial_loss_mode="fixed",
            initial_loss_frac=float(shock_frac),
            spread_without_default=True,
        )
        sim_rows.append({
            "bank_id":           int(bank_id),
            "shock_fraction":    result["initial_shock_fraction"],
            "initial_default":   int(result["initial_default"]),
            "cascade_num_failed": result["num_failed"],
            "cascade_total_loss": result["total_loss"],
            "cascade_rounds":    result["rounds"],
        })

    df_sim = pd.DataFrame(sim_rows)
    df_sim["year"]    = year
    df_sim["quarter"] = q

    out_path = OUT_SIMULATION / f"simulation_features_{year}Q{q}.parquet"
    df_sim.to_parquet(out_path, index=False)

    simulation_summary_rows.append({
        "year":                  year,
        "quarter":               q,
        "period":                f"{year} Q{q}",
        "n_simulations":         N_SIMULATIONS,
        "pct_initial_default":   df_sim["initial_default"].mean(),
        "avg_cascade_failed":    df_sim["cascade_num_failed"].mean(),
        "avg_cascade_total_loss":df_sim["cascade_total_loss"].mean(),
        "saved_file":            out_path.name,
    })

    print(f"[OK] {year}Q{q} | {N_SIMULATIONS} sims | saved {out_path.name}")

df_simulation_summary = (
    pd.DataFrame(simulation_summary_rows)
      .sort_values(["year", "quarter"])
      .reset_index(drop=True)
)

display(df_simulation_summary)

In [ ]:
# Quick verification: inspect the saved parquet for one quarter
df_sim_check = pd.read_parquet(OUT_SIMULATION / "simulation_features_2016Q1.parquet")

display(df_sim_check.head())
print(df_sim_check.shape)
df_sim_check[["shock_fraction", "initial_default", "cascade_num_failed", "cascade_total_loss", "cascade_rounds"]].describe()